# 🧠 基于 Nilearn 的电极筛选策略多维可视化

本 Notebook 对被试 `test001`、`test002`、`test003` 在不同筛选策略下提取出的电极进行 MNI 空间皮层投影可视化。

### 可视化规则：
1. **筛选策略与颜色**：
   * 🟢 **绿色 (Green)**: 策略 1 (混合 100-400ms 平均显著 + 靶区)
   * 🔵 **蓝色 (Blue)**: 策略 2 (混合 连续 50ms 显著 + 靶区)
   * 🟣 **紫色 (Purple)**: 策略 3 (单一 100-400ms 平均显著 + 靶区)
   * 🟡 **黄色 (Yellow)**: 策略 4 (单一 连续 50ms 显著 + 靶区)
   * *注：如果一个电极同时满足多个策略，将以其所能达到的**最高策略级别**（即策略 1 > 策略 2 > 策略 3 > 策略 4）进行着色。*

2. **信号特征与标记形状**：
   * **实心圆点 (Solid Circles)**: ERP 筛选到的电极
   * **空心圆圈 (Hollow Circles)**: High Gamma (60-150Hz) 筛选到的电极

3. **解剖限制（靶区定义）**：
   * **枕叶**: Calcarine, Occipital_Inf, Occipital_Mid, Lingual
   * **颞叶下/后**: Fusiform, Temporal_Inf
   * **颞叶前/上**: Temporal_Mid, Temporal_Pole

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import scipy.io as sio
from pymatreader import read_mat
from scipy.stats import ranksums
import matplotlib.pyplot as plt
from nilearn import plotting

base_dir = '/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'
subjects = ['test001', 'test002', 'test003']

## 1. 定义电极筛选与脑区提取函数

In [ ]:
def get_electrode_anatomies(subject):
    loc_path = os.path.join(base_dir, 'processed_data', subject, f'{subject}_ieegloc.xlsx')
    if not os.path.exists(loc_path):
        return {}
    df = pd.read_excel(loc_path)
    cols = df.columns.tolist()
    dk_col = 'Desikan-Killiany' if 'Desikan-Killiany' in cols else ('Desikan-Killiany_prob' if 'Desikan-Killiany_prob' in cols else '')
    dkt_col = 'DKT' if 'DKT' in cols else ''
    aal_col = 'AAL3 (MNI-linear)' if 'AAL3 (MNI-linear)' in cols else ('AAL3 (MNI-segment)' if 'AAL3 (MNI-segment)' in cols else '')
    
    anat_dict = {}
    target_terms = ['calcarine', 'occipital_inf', 'occipital_mid', 'lingual', 'fusiform', 'temporal_inf', 'temporal_mid', 'temporal_pole']
    
    for idx, row in df.iterrows():
        ch = str(row['Channel'])
        in_target = False
        for col in [dk_col, dkt_col, aal_col]:
            if col and pd.notna(row[col]):
                val = str(row[col]).lower().replace('-', '_').replace(' ', '_')
                if any(term in val for term in target_terms):
                    in_target = True
                    break
        anat_dict[ch] = in_target
    return anat_dict

def get_mni_coordinates(subject):
    loc_path = os.path.join(base_dir, 'processed_data', subject, f'{subject}_ieegloc.xlsx')
    if not os.path.exists(loc_path):
        return {}
    df = pd.read_excel(loc_path)
    mni_dict = {}
    for idx, row in df.iterrows():
        ch = str(row['Channel'])
        if 'MNI' in df.columns and pd.notna(row['MNI']):
            try:
                coords = ast.literal_eval(str(row['MNI']))
                mni_dict[ch] = coords
            except:
                pass
    return mni_dict

## 2. 执行统计筛选与 MNI 坐标定位

In [ ]:
def process_subject_electrodes(subject, feature_type):
    anat_map = get_electrode_anatomies(subject)
    mni_map = get_mni_coordinates(subject)
    
    # 确定文件路径
    if feature_type == 'erp':
        mat_path = os.path.join(base_dir, 'processed_data', subject, 'task1_ERP_epoched.mat')
    else:
        if subject == 'test001':
            mat_path = os.path.join(base_dir, 'color_cognition_pipeline', 'feature', 'subband_60_150', 'task1_hg_subband.mat')
        else:
            mat_path = os.path.join(base_dir, 'color_cognition_pipeline', 'feature', 'subband_60_150', subject, 'task1_hg_subband.mat')
            
    if not os.path.exists(mat_path):
        return {}
        
    if feature_type == 'erp':
        mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
        epoch = mat['epoch']
        data = epoch.data
        ch_names = [str(ch.labels) for ch in epoch.ch]
        time_ms = epoch.time_ms if hasattr(epoch, 'time_ms') else np.linspace(-500, 998, data.shape[-1])
        data_cell = None
    else:
        mat = read_mat(mat_path)
        epoch = mat['epoch']
        data_cell = epoch['data_cell']
        ch_names = epoch['ch']['labels']
        if isinstance(ch_names, str): ch_names = [ch_names]
        ch_names = [str(c) for c in ch_names]
        time_ms = epoch['time_ms']
        data = None
        
    t_100_400 = np.where((time_ms >= 100) & (time_ms <= 400))[0]
    t_50_400 = np.where((time_ms >= 50) & (time_ms <= 400))[0]
    dt = time_ms[1] - time_ms[0]
    win_len = int(np.ceil(50.0 / dt))
    
    cond_pairs = [(0, 1), (2, 3), (4, 5), (6, 7)]
    merged_c_idx = [0, 2, 4, 6]
    merged_g_idx = [1, 3, 5, 7]
    
    elec_strategies = {}
    
    for ch_idx, elec in enumerate(ch_names):
        # 只考虑靶区内且含有 MNI 坐标的电极
        if not anat_map.get(elec, False) or elec not in mni_map:
            continue
            
        if feature_type == 'erp':
            c_merged = np.concatenate([data[idx, :, ch_idx, :] for idx in merged_c_idx], axis=0)
            g_merged = np.concatenate([data[idx, :, ch_idx, :] for idx in merged_g_idx], axis=0)
            single_c = [data[pair[0], :, ch_idx, :] for pair in cond_pairs]
            single_g = [data[pair[1], :, ch_idx, :] for pair in cond_pairs]
        else:
            c_merged = np.concatenate([data_cell[idx][:, ch_idx, :] for idx in merged_c_idx], axis=0)
            g_merged = np.concatenate([data_cell[idx][:, ch_idx, :] for idx in merged_g_idx], axis=0)
            single_c = [data_cell[pair[0]][:, ch_idx, :] for pair in cond_pairs]
            single_g = [data_cell[pair[1]][:, ch_idx, :] for pair in cond_pairs]
            
        c_merged = c_merged[~np.isnan(c_merged).any(axis=1)]
        g_merged = g_merged[~np.isnan(g_merged).any(axis=1)]
        if len(c_merged) == 0 or len(g_merged) == 0: continue
        
        single_c = [arr[~np.isnan(arr).any(axis=1)] for arr in single_c]
        single_g = [arr[~np.isnan(arr).any(axis=1)] for arr in single_g]
        
        # 策略 1: 混合 100-400ms 平均
        c_m_avg = np.mean(c_merged[:, t_100_400], axis=1)
        g_m_avg = np.mean(g_merged[:, t_100_400], axis=1)
        stat1, p1 = ranksums(c_m_avg, g_m_avg)
        s1 = (p1 < 0.05 and stat1 > 0)
        
        # 策略 2: 混合 连续 50ms
        sig_arr = np.zeros(len(t_50_400))
        for i, t in enumerate(t_50_400):
            stat2, p2 = ranksums(c_merged[:, t], g_merged[:, t])
            if p2 < 0.05 and stat2 > 0: sig_arr[i] = 1
        s2 = False
        count_len = 0
        for val in sig_arr:
            if val == 1:
                count_len += 1
                if count_len >= win_len: s2 = True; break
            else: count_len = 0
            
        # 策略 3: 单一 100-400ms 平均
        s3 = False
        for c_arr, g_arr in zip(single_c, single_g):
            if len(c_arr) == 0 or len(g_arr) == 0: continue
            c_s_avg = np.mean(c_arr[:, t_100_400], axis=1)
            g_s_avg = np.mean(g_arr[:, t_100_400], axis=1)
            stat3, p3 = ranksums(c_s_avg, g_s_avg)
            if p3 < 0.05 and stat3 > 0: s3 = True; break
            
        # 策略 4: 单一 连续 50ms
        s4 = False
        for c_arr, g_arr in zip(single_c, single_g):
            if len(c_arr) == 0 or len(g_arr) == 0: continue
            sig_arr_s = np.zeros(len(t_50_400))
            for i, t in enumerate(t_50_400):
                stat4, p4 = ranksums(c_arr[:, t], g_arr[:, t])
                if p4 < 0.05 and stat4 > 0: sig_arr_s[i] = 1
            count_len_s = 0
            for val in sig_arr_s:
                if val == 1:
                    count_len_s += 1
                    if count_len_s >= win_len: s4 = True; break
                else: count_len_s = 0
            if s4: break
            
        # 决定最高策略等级
        best_strat = None
        if s1: best_strat = 1
        elif s2: best_strat = 2
        elif s3: best_strat = 3
        elif s4: best_strat = 4
        
        if best_strat is not None:
            elec_strategies[elec] = {
                'strategy': best_strat,
                'coords': mni_map[elec]
            }
    return elec_strategies

## 3. 汇总三个被试的坐标与颜色映射

In [ ]:
# 策略与颜色的对应关系
strategy_colors = {
    1: '#2ca02c',  # 🟢 绿色 (Strategy 1)
    2: '#1f77b4',  # 🔵 蓝色 (Strategy 2)
    3: '#9467bd',  # 🟣 紫色 (Strategy 3)
    4: '#ff7f0e'   # 🟡 橙/黄色 (Strategy 4)
}

erp_coords, erp_colors, erp_labels = [], [], []
hg_coords, hg_colors, hg_labels = [], [], []

for subj in subjects:
    erp_dict = process_subject_electrodes(subj, 'erp')
    for elec, info in erp_dict.items():
        erp_coords.append(info['coords'])
        erp_colors.append(strategy_colors[info['strategy']])
        erp_labels.append(f"{subj}_{elec}_S{info['strategy']}")
        
    hg_dict = process_subject_electrodes(subj, 'highgamma')
    for elec, info in hg_dict.items():
        hg_coords.append(info['coords'])
        hg_colors.append(strategy_colors[info['strategy']])
        hg_labels.append(f"{subj}_{elec}_S{info['strategy']}")
        
print(f"已提取 ERP 电极共 {len(erp_coords)} 个。")
print(f"已提取 High Gamma 电极共 {len(hg_coords)} 个。")

## 4. 2D 投影图可视化 (ERP: 实心, HG: 空心)

In [ ]:
# 初始化全脑玻璃脑投影面
fig = plt.figure(figsize=(15, 10))
display = plotting.plot_glass_brain(None, display_mode='ortho', figure=fig, title='Electrode Selection (ERP: Solid, HG: Hollow)')

# 1. 叠加 ERP 电极 (实心圆点)
if erp_coords:
    display.add_markers(
        marker_coords=np.array(erp_coords),
        marker_color=erp_colors,
        marker_size=120,
        marker='o',
        alpha=0.95
    )

# 2. 叠加 High Gamma 电极 (无填充空心圆圈，利用 edgecolors 和 marker_color='none')
if hg_coords:
    display.add_markers(
        marker_coords=np.array(hg_coords),
        marker_color='none',
        edgecolors=hg_colors,
        marker_size=120,
        linewidths=2.5,
        marker='o',
        alpha=0.95
    )

# 绘制图例
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=10, label='Strategy 1 (Merged Mean)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=10, label='Strategy 2 (Merged Cont 50ms)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#9467bd', markersize=10, label='Strategy 3 (Single Mean)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff7f0e', markersize=10, label='Strategy 4 (Single Cont 50ms)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='ERP (Solid Circle)'),
    Line2D([0], [0], marker='o', color='gray', markerfacecolor='none', markeredgewidth=2, markersize=10, label='High Gamma (Hollow Circle)')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.05))
plt.show()

## 5. 3D 交互式可视化图

In [ ]:
# 使用 view_markers 分别为 ERP 与 High Gamma 生成可缩放、可旋转的 3D 脑表面交互式图

print("正在渲染 ERP 电极 3D 交互视图...")
view_erp = plotting.view_markers(
    marker_coords=np.array(erp_coords),
    marker_color=erp_colors,
    marker_size=12,
    title="ERP Electrode Selection - 3D View"
)

print("正在渲染 High Gamma 电极 3D 交互视图...")
view_hg = plotting.view_markers(
    marker_coords=np.array(hg_coords),
    marker_color=hg_colors,
    marker_size=12,
    title="High Gamma Electrode Selection - 3D View"
)

在 Jupyter Notebook 中，您可以通过直接执行下方的单元格来内嵌展示交互界面。并且我们已经将这 2 个交互脑图保存为了 `.html` 格式，您可以在网页浏览器中双击文件进行全屏交互（如旋转、缩放、鼠标悬浮显示电极信息）。

In [ ]:
# 导出为独立的 HTML 文件
html_dir = os.path.join(base_dir, 'color_cognition_pipeline', 'images', 'interactive_brain')
os.makedirs(html_dir, exist_ok=True)

erp_html_path = os.path.join(html_dir, 'erp_3d_brain.html')
hg_html_path = os.path.join(html_dir, 'hg_3d_brain.html')

view_erp.save_as_html(erp_html_path)
view_hg.save_as_html(hg_html_path)

print(f"ERP 3D 交互视图已保存至：{erp_html_path}")
print(f"High Gamma 3D 交互视图已保存至：{hg_html_path}")

# 在 Notebook 中直接展示 ERP 视图
view_erp

In [ ]:
# 在 Notebook 中直接展示 High Gamma 视图
view_hg

## 6. 针对 Object 条件 (Object Color vs. Object Gray) 的显著性分析与可视化

在之前筛选出的枕叶与颞叶电极库中，进一步考察对 **Object 类别本身** 具备显著性颜色选择性的电极。筛选标准：
1. **100-400ms 平均值显著**
2. **50-400ms 期间存在连续 50ms 以上的显著窗口**

In [ ]:
# 预选电极库 (已限靶区且对任一类别颜色显著的电极集)
selected_electrodes = {
    'erp': {
        'test001': ['B5', 'C8', 'C10', 'F9', 'G11', 'H1', 'H9'],
        'test002': ['A3', 'B1', 'C7', 'C8', 'F3', 'F5'],
        'test003': ['A12', 'G11', 'G12']
    },
    'highgamma': {
        'test001': ['B5', 'C10', 'E8', 'G4', 'H6', 'H9', 'H10'],
        'test002': ['A2', 'A3', 'A4', 'A8', 'A9', 'B6', 'C9', 'C10', 'F4', 'F6', 'G5', 'G6', 'G7', 'H5', 'H8'],
        'test003': ['D9', 'G9', 'G10', 'G11', 'G14']
    }
}

color_map = {
    'mean_only': '#1f77b4',  # 🔵 蓝色
    'win_only': '#2ca02c',   # 🟢 绿色
    'both': '#d62728'        # 🔴 红色
}

object_results = []

for feature_type in ['erp', 'highgamma']:
    for subj in subjects:
        mni_map = get_mni_coordinates(subj)
        elec_list = selected_electrodes[feature_type][subj]
        if not elec_list: continue
        
        if feature_type == 'erp':
            mat_path = os.path.join(base_dir, 'processed_data', subj, 'task1_ERP_epoched.mat')
        else:
            if subj == 'test001':
                mat_path = os.path.join(base_dir, 'color_cognition_pipeline', 'feature', 'subband_60_150', 'task1_hg_subband.mat')
            else:
                mat_path = os.path.join(base_dir, 'color_cognition_pipeline', 'feature', 'subband_60_150', subj, 'task1_hg_subband.mat')
                
        if not os.path.exists(mat_path): continue
        
        if feature_type == 'erp':
            mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
            epoch = mat['epoch']
            data = epoch.data
            ch_names = [str(ch.labels) for ch in epoch.ch]
            time_ms = epoch.time_ms if hasattr(epoch, 'time_ms') else np.linspace(-500, 998, data.shape[-1])
            data_cell = None
        else:
            mat = read_mat(mat_path)
            epoch = mat['epoch']
            data_cell = epoch['data_cell']
            ch_names = epoch['ch']['labels']
            if isinstance(ch_names, str): ch_names = [ch_names]
            ch_names = [str(c) for c in ch_names]
            time_ms = epoch['time_ms']
            data = None
            
        t_100_400 = np.where((time_ms >= 100) & (time_ms <= 400))[0]
        t_50_400 = np.where((time_ms >= 50) & (time_ms <= 400))[0]
        dt = time_ms[1] - time_ms[0]
        win_len = int(np.ceil(50.0 / dt))
        
        for elec in elec_list:
            if elec not in ch_names or elec not in mni_map:
                continue
            ch_idx = ch_names.index(elec)
            coords = mni_map[elec]
            
            if feature_type == 'erp':
                c_data = data[2, :, ch_idx, :]
                g_data = data[3, :, ch_idx, :]
            else:
                c_data = data_cell[2][:, ch_idx, :]
                g_data = data_cell[3][:, ch_idx, :]
                
            c_data = c_data[~np.isnan(c_data).any(axis=1)]
            g_data = g_data[~np.isnan(g_data).any(axis=1)]
            if len(c_data) == 0 or len(g_data) == 0: continue
            
            # 1. 均值
            c_avg = np.mean(c_data[:, t_100_400], axis=1)
            g_avg = np.mean(g_data[:, t_100_400], axis=1)
            stat_m, p_m = ranksums(c_avg, g_avg)
            m_sig = (p_m < 0.05 and stat_m > 0)
            
            # 2. 连续
            sig_arr = np.zeros(len(t_50_400))
            for i, t in enumerate(t_50_400):
                stat_w, p_w = ranksums(c_data[:, t], g_data[:, t])
                if p_w < 0.05 and stat_w > 0: sig_arr[i] = 1
            w_sig = False
            count_len = 0
            max_len = 0
            for val in sig_arr:
                if val == 1:
                    count_len += 1
                    if count_len > max_len: max_len = count_len
                    if count_len >= win_len: w_sig = True
                else:
                    count_len = 0
                    
            if m_sig or w_sig:
                strat = 'both' if (m_sig and w_sig) else ('mean_only' if m_sig else 'win_only')
                strat_desc = '均值与连续窗均显著' if strat == 'both' else ('仅100-400ms均值显著' if strat == 'mean_only' else '仅连续50ms以上显著')
                object_results.append({
                    'subject': subj,
                    'electrode': elec,
                    'type': feature_type.upper(),
                    'coords': coords,
                    'mean_sig': m_sig,
                    'win_sig': w_sig,
                    'p_mean': p_m if m_sig else np.nan,
                    'max_win_ms': max_len * dt,
                    'strategy': strat,
                    'strategy_desc': strat_desc,
                    'color': color_map[strat]
                })

df_obj = pd.DataFrame(object_results)
print(f"共筛选出对 Object 显著的电极 {len(object_results)} 个。")
df_obj[['subject', 'electrode', 'type', 'mean_sig', 'win_sig', 'strategy_desc']]

### 6.1 Object 显著电极 2D 全脑玻璃投影

In [ ]:
fig = plt.figure(figsize=(15, 10))
display = plotting.plot_glass_brain(None, display_mode='ortho', figure=fig, title='Object-Specific Color Electrodes')

erp_o_coords = [r['coords'] for r in object_results if r['type'] == 'ERP']
erp_o_colors = [r['color'] for r in object_results if r['type'] == 'ERP']
hg_o_coords = [r['coords'] for r in object_results if r['type'] == 'HIGHGAMMA']
hg_o_colors = [r['color'] for r in object_results if r['type'] == 'HIGHGAMMA']

if erp_o_coords:
    display.add_markers(
        marker_coords=np.array(erp_o_coords),
        marker_color=erp_o_colors,
        marker_size=150,
        marker='o',
        alpha=0.95
    )
if hg_o_coords:
    display.add_markers(
        marker_coords=np.array(hg_o_coords),
        marker_color='none',
        edgecolors=hg_o_colors,
        marker_size=150,
        linewidths=3.0,
        marker='o',
        alpha=0.95
    )

from matplotlib.lines import Line2D
legend_elements_o = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markersize=10, label='Both Mean & Window Significant (Red)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=10, label='Only 100-400ms Mean Significant (Blue)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=10, label='Only Continuous >50ms Window Significant (Green)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='ERP (Solid Circle)'),
    Line2D([0], [0], marker='o', color='gray', markerfacecolor='none', markeredgewidth=2, markersize=10, label='High Gamma (Hollow Circle)')
]
fig.legend(handles=legend_elements_o, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.05))
plt.show()

### 6.2 Object 显著电极 3D 脑表面投影

In [ ]:
if erp_o_coords:
    view_erp_o = plotting.view_markers(np.array(erp_o_coords), marker_color=erp_o_colors, marker_size=15, title='Object Selective ERP')
else:
    view_erp_o = None
    
if hg_o_coords:
    view_hg_o = plotting.view_markers(np.array(hg_o_coords), marker_color=hg_o_colors, marker_size=15, title='Object Selective High Gamma')
else:
    view_hg_o = None

if view_erp_o:
    erp_html_o = os.path.join(base_dir, 'color_cognition_pipeline', 'images', 'interactive_brain', 'erp_3d_object_brain.html')
    view_erp_o.save_as_html(erp_html_o)
if view_hg_o:
    hg_html_o = os.path.join(base_dir, 'color_cognition_pipeline', 'images', 'interactive_brain', 'hg_3d_object_brain.html')
    view_hg_o.save_as_html(hg_html_o)
    
view_hg_o